# AI4RAG William benchmark — PR #75 (fix-prompts)

Compare prompt templates on the **same** GAM search space and William benchmark (22 questions).

**Run order:** `ai4rag_baseline_experiment.ipynb` → `ai4rag_pr75_experiment.ipynb`.

**Inspect without re-run:** set `RUN_EXPERIMENT = False` in section 2 and run from section 4 onward (loads `results/{variant}/runs/<latest>/`).

**Prerequisites:** OGX `.env`, William data in `../lightrag/POC/challenge_data/`, ~10–20 min if running GAM.

**Branch:** `fix-prompts`

## 1. Install ai4rag

Skip if inspecting a saved run only (`RUN_EXPERIMENT = False`).

In [1]:
# Install ai4rag from: fix-prompts
!pip install --force-reinstall git+https://github.com/IBM/ai4rag.git@fix-prompts --quiet
!pip install python-dotenv openai pandas docling docling-core --quiet

print("Installed ai4rag @ fix-prompts")

Installed ai4rag @ fix-prompts


## 2. Configuration

In [2]:
import json
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('.').resolve()))
import experiment_utils as eu

VARIANT = "pr75"
BRANCH = "fix-prompts"
PR_URL = "https://github.com/IBM/ai4rag/pull/75"
EXPECT_PR75 = True

# --- execution controls ---
RUN_EXPERIMENT = True   # False = load latest saved run (no GAM / no OGX experiment)
RUN_ID = None           # e.g. '20260623_163137' or None for latest
RUN_LLM_JUDGE = True    # 220 API calls; set False to skip or when reloading saved run

eu.print_saved_run_index(VARIANT)

if RUN_EXPERIMENT:
    env_path = eu.load_dotenv_from_standard_paths()
    print(f'Env: {env_path or "environment variables"}')
    OGX_BASE_URL, OGX_API_KEY = eu.require_ogx_credentials()
    print(f'OGX: {OGX_BASE_URL}')
    try:
        import ai4rag
        print(f'ai4rag version: {getattr(ai4rag, "__version__", "unknown")}')
    except ImportError as exc:
        raise ImportError('Run the install cell first') from exc
else:
    print('RUN_EXPERIMENT=False — will load persisted results in section 4')

Saved experiment runs

pr75/ (1 runs, latest=20260624_072341)
  20260624_072341  best_faith=52.6% ← latest
Env: ../lightrag/POC/.env
OGX: https://<OGX_BASE_URL>
ai4rag version: 0.8.0


## 3. Load William benchmark

Skipped when `RUN_EXPERIMENT = False`.

In [3]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('.').resolve()))
import experiment_utils as eu
eu.ensure_notebook_context("pr75")

if RUN_EXPERIMENT:
    benchmark_data, documents = eu.load_william_benchmark()
    benchmark_df = pd.DataFrame(benchmark_data)
    print(f'Questions: {len(benchmark_data)}, documents: {len(documents)}')
else:
    benchmark_data, documents, benchmark_df = [], [], pd.DataFrame()
    print('Skipped (loading saved run)')

Questions: 22, documents: 10


## 4. Run experiment or load saved run

Fresh run: GAM explores retrieval/chunking from ai4rag defaults. Reload: reads `results/{variant}/runs/<run_id>/` (summary, leaderboard, answers, patterns, prompts).

In [4]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('.').resolve()))
import experiment_utils as eu
eu.ensure_notebook_context("pr75")

prompt_info = None

if RUN_EXPERIMENT:
    from ai4rag.rag.foundation_models.ogx import OgxClient

    ogx_client = OgxClient(base_url=OGX_BASE_URL, api_key=OGX_API_KEY)
    docling_documents = eu.documents_to_docling(documents)
    print(f'Converted {len(docling_documents)} documents')

    experiment = eu.run_gam_experiment(
        docling_documents, benchmark_df, ogx_client, label='PR #75',
    )
    experiment_results, all_answers = eu.extract_experiment_results(experiment, benchmark_data)
    eu.enrich_results_with_answer_quality(experiment_results, all_answers)
    if experiment.results:
        top = experiment.results.get_best_evaluations(k=1)[0]
        prompt_info = eu.print_prompt_verification(top, expect_pr75=EXPECT_PR75)
    print('Experiment complete')
else:
    bundle = eu.load_saved_run(VARIANT, run_id=RUN_ID)
    experiment_results = bundle['experiment_results']
    all_answers = bundle['all_answers']
    leaderboard_df = bundle['leaderboard_df']
    prompt_info = bundle.get('prompts')
    run_dir = bundle['run_dir']
    print(f'Loaded saved run: {run_dir}')
    eu.print_run_inspection(bundle)

   [info] evaluation: Evaluating the RAG Pattern 'Pattern9' response using UnitxtEvaluator.
   🔧 Pattern unknown created
System message (first 200 chars):
You are a retrieval-augmented assistant. Answer using ONLY the provided documents. You are a helpful, respectful and honest assistant. Always answer as helpfully as possible, while being safe. Your an...

User template (first 300 chars):
Answer ONLY using information from the documents below. Do not use outside knowledge. If the documents do not contain the answer, say you do not have enough information.
You MUST cite sources using [1], [2], etc. matching the document numbers for every factual claim.

Documents:
{reference_documents...

Context template:
Document {doc_number}:
{document}


PR #75 feature checklist:
  ✅ strong grounding
  ✅ mandatory citations
  ✅ english enforcement
  ✅ numbered documents

✅ Prompt verification OK for PR #75 (4/4 PR features).
Experiment complete


## 5. LLM-as-a-Judge

Skipped if answers already include `llm_judge` from a saved run.

In [5]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('.').resolve()))
import experiment_utils as eu
eu.ensure_notebook_context("pr75")

has_judge = all_answers and all('llm_judge' in a for a in all_answers)

if RUN_LLM_JUDGE and all_answers and not has_judge:
    if not RUN_EXPERIMENT:
        raise RuntimeError('LLM judge needs OGX — set RUN_EXPERIMENT=True or use a saved run that already has llm_judge')
    judge_fn = eu.create_llm_judge_fn(OGX_BASE_URL, OGX_API_KEY)
    eu.run_llm_judge_on_answers(all_answers, judge_fn)
    eu.attach_llm_judge_averages(experiment_results, all_answers)
    print('LLM-as-a-Judge complete')
elif has_judge:
    eu.attach_llm_judge_averages(experiment_results, all_answers)
    print('Using llm_judge scores from saved run')
else:
    print('Skipped LLM-as-a-Judge')

0.80
LLM-as-a-Judge complete


## 6. Leaderboard

In [6]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('.').resolve()))
import experiment_utils as eu
eu.ensure_notebook_context("pr75")

if 'leaderboard_df' not in globals() or leaderboard_df is None or leaderboard_df.empty:
    leaderboard_df = eu.build_leaderboard_df(experiment_results)

display_df = eu.format_leaderboard_for_display(leaderboard_df)
print("AI4RAG PR #75 LEADERBOARD")
print('=' * 100)
print(display_df.to_string(index=False))

if not leaderboard_df.empty:
    run_summary = eu.summarize_run(experiment_results, all_answers)
    print('\nRun summary:')
    for k, v in run_summary.items():
        print(f'  {k}: {v:.1%}')

AI4RAG PR #75 LEADERBOARD
 Pattern Retrieval  Window  Chunks Chunking Chunk Size Faithfulness Answer Correctness LLM Judge Citation Rate Multilingual % Combined  Final Score
Pattern4    simple       0       3      N/A        N/A        55.2%              62.2%     82.7%         72.7%           9.1%    58.7%       0.5520
Pattern3    window       3       5      N/A        N/A        47.4%              59.3%     80.0%         81.8%           9.1%    53.3%       0.4738
Pattern8    window       5       3      N/A        N/A        46.8%              60.7%     83.6%         86.4%           4.5%    53.8%       0.4683
Pattern2    window       5      10      N/A        N/A        46.1%              62.5%     83.6%         81.8%           4.5%    54.3%       0.4606
Pattern5    window       3      10      N/A        N/A        45.4%              62.1%     80.9%         81.8%           9.1%    53.8%       0.4539
Pattern7    window       1      10      N/A        N/A        45.0%              61.7%

## 7. Save artifacts

Writes a full run folder:
`results/pr75/runs/<timestamp>/` with `summary.json`, `leaderboard.csv`, `answers.csv`, `patterns.json`, `prompts.json`.

Skipped when reloading (`RUN_EXPERIMENT = False`).

In [11]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('.').resolve()))
import experiment_utils as eu
eu.ensure_notebook_context("pr75")

if RUN_EXPERIMENT and experiment_results:
    run_dir = eu.save_experiment_artifacts(
        VARIANT,
        branch=BRANCH,
        pr_url=PR_URL,
        benchmark_data=benchmark_data,
        experiment_results=experiment_results,
        all_answers=all_answers,
        leaderboard_df=leaderboard_df,
        prompt_info=prompt_info if isinstance(prompt_info, dict) else None,
    )
    print(f'Saved run: {run_dir}')
    print('Files: summary.json, leaderboard.csv, answers.csv, patterns.json, prompts.json')
else:
    print('Skipped save (no new experiment run)')

Saved run: /Users/lcmielow/Documents/GitHub/prototypes/AutoRAG/comparative_analysis/autorag/results/pr75/runs/20260624_082508
Files: summary.json, leaderboard.csv, answers.csv, patterns.json, prompts.json


## 8. Inspect saved runs (offline)

Browse all runs or load a specific `RUN_ID` without re-running GAM.

In [12]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('.').resolve()))
import experiment_utils as eu
eu.ensure_notebook_context("pr75")

eu.print_saved_run_index(VARIANT)

# Uncomment to inspect a specific run:
# inspect = eu.load_saved_run(VARIANT, run_id='20260623_163137')
# eu.print_run_inspection(inspect)
# inspect['answers_df'].head(10)

Saved experiment runs

pr75/ (3 runs, latest=20260624_082508)
  20260624_082508  best_faith=55.2% ← latest
  20260624_081751  best_faith=55.2%
  20260624_072341  best_faith=52.6%


## 9. Compare with baseline

In [14]:
import importlib
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('.').resolve()))
import experiment_utils as eu
eu = importlib.reload(eu)
eu.ensure_notebook_context("pr75")

baseline_summary = eu.load_summary(eu.BASELINE_SUMMARY_LATEST)
pr75_summary = eu.load_summary(eu.PR75_SUMMARY_LATEST)

if baseline_summary and pr75_summary:
    eu.print_baseline_comparison(pr75_summary, baseline_summary)
else:
    print('Run baseline notebook first (section 7) to enable comparison.')
    if not baseline_summary:
        print(f'  Missing: {eu.BASELINE_SUMMARY_LATEST}')


📈 PAIRED COMPARISON (baseline notebook vs PR #75 notebook)
--------------------------------------------------------------------------------
Metric                 |   Baseline |     PR #75 |    Delta
--------------------------------------------------------------------------------
Best faithfulness      |     43.3% |     55.2% |  +11.9 pp
Mean faithfulness      |     34.3% |     46.3% |  +11.9 pp
Best answer corr.      |     35.6% |     62.5% |  +26.9 pp
Mean answer corr.      |     29.3% |     61.6% |  +32.3 pp
Best LLM judge         |     78.2% |     83.6% |   +5.5 pp
Mean LLM judge         |     74.3% |     82.0% |   +7.7 pp
Citation rate          |      0.0% |     82.3% |  +82.3 pp
Multilingual rate      |     55.9% |      7.6% |  -48.3 pp
--------------------------------------------------------------------------------

Note: GAM explores different retrieval configs each run. For strict prompt-only A/B, fix rag_params and re-run both branches.
